In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import time
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive', force_remount=True)

# ==========================================
# CLASE DE MONITOREO DE RECURSOS
# ==========================================
class Monitor:
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.start_time = 0
        self.start_cpu = 0

    def iniciar(self):
        self.start_time = time.time()
        self.start_cpu = self.process.cpu_percent()

    def detener(self):
        end_time = time.time()
        end_cpu = self.process.cpu_percent()
        mem_info = self.process.memory_info()

        tiempo_total = end_time - self.start_time
        uso_cpu = (end_cpu + self.start_cpu) / 2 # Promedio simple
        uso_ram_mb = mem_info.rss / 1024 / 1024 # Convertir Bytes a MB

        return tiempo_total, uso_cpu, uso_ram_mb

# ==========================================
# CONFIGURACIÓN
# ==========================================
# Rutas (Ajusta si es necesario)
PATH_S = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/S_Dataset_Unificado_FINAL.csv'
PATH_TOTAL = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'

# Función auxiliar para buscar archivos
def buscar_archivo(nombre):
    f = glob.glob(os.path.join('/content/drive/MyDrive/DOCTORADO', "**", f"*{nombre}*"), recursive=True)
    return f[0] if f else None

PATH_MM = buscar_archivo("Multi-Modal_Intelligent")
PATH_VEH = buscar_archivo("VehicularData")

# ==========================================
# MOTOR DE ENTRENAMIENTO Y METRICAS AVANZADAS
# ==========================================
def evaluar_escenario(ruta_csv, nombre_escenario):
    print(f"\n🔬 --- EVALUANDO: {nombre_escenario} ---")
    monitor = Monitor()
    metrics = {'Scenario': nombre_escenario}

    # 1. CARGA DE DATOS
    try:
        df = pd.read_csv(ruta_csv)
        # Limpieza rápida por si acaso
        if 'Source' in df.columns:
            # Si es el unificado total, filtramos por source si quisiéramos probar individualmente,
            # pero aquí asumimos que la ruta_csv es el dataset completo a probar.
            pass

        req_cols = ['latitude', 'longitude', 'speed', 'course']
        df = df[req_cols].dropna().reset_index(drop=True)

        if len(df) > 50000: df = df.iloc[:50000] # LIMITAMOS A 50k para no saturar RAM en pruebas

    except Exception as e:
        print(f"❌ Error cargando {nombre_escenario}: {e}")
        return None

    # 2. PREPARACIÓN (Time Steps)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df)

    TIME_STEPS = 30
    def create_seq(data):
        xs = []
        for i in range(len(data)-TIME_STEPS):
            xs.append(data[i:(i+TIME_STEPS)])
        return np.array(xs)

    X = create_seq(X_scaled)
    if len(X) < 100: return None

    # Split 80/20
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]

    # 3. ENTRENAMIENTO (Midiendo Recursos)
    print("   🧠 Entrenando LSTM...")
    model = Sequential([
        LSTM(64, activation='relu', input_shape=(TIME_STEPS, 4), return_sequences=False),
        RepeatVector(TIME_STEPS),
        LSTM(64, activation='relu', return_sequences=True),
        TimeDistributed(Dense(4))
    ])
    model.compile(optimizer='adam', loss='mse')

    monitor.iniciar()
    history = model.fit(X_train, X_train, epochs=3, batch_size=64, verbose=0)
    t_train, cpu_train, ram_train = monitor.detener()

    metrics['Train Time (s)'] = round(t_train, 2)
    metrics['Train CPU (%)'] = round(cpu_train, 2)
    metrics['Train RAM (MB)'] = round(ram_train, 2)

    # 4. CÁLCULO DE UMBRAL (MAE en Train)
    X_train_pred = model.predict(X_train, verbose=0)
    train_mae = np.mean(np.abs(X_train_pred - X_train), axis=(1,2))
    umbral = np.mean(train_mae) + 3 * np.std(train_mae)
    metrics['MAE Threshold'] = round(umbral, 5)
    metrics['Normal MAE Avg'] = round(np.mean(train_mae), 5)

    # 5. VALIDACIÓN (Simulación de Ataque)
    # Simular ataque en el 50% final del test set
    print("   ⚔️ Simulando Ataque y Calculando Métricas...")
    monitor.iniciar()

    # Datos limpios
    X_test_pred = model.predict(X_test, verbose=0)
    test_mae = np.mean(np.abs(X_test_pred - X_test), axis=(1,2))

    # Crear vector de etiquetas reales (Ground Truth)
    # 0 = Normal, 1 = Ataque
    y_true = np.zeros(len(test_mae))
    punto_ataque = len(test_mae) // 2
    y_true[punto_ataque:] = 1 # La segunda mitad es "Ataque"

    # Inyectar anomalía matemática en el MAE de la segunda mitad
    # (Simulamos que el modelo detecta un error alto)
    # Nota: En un caso real, la data de entrada vendría alterada.
    # Aquí forzamos el error para probar las métricas de clasificación.
    factor_anomalia = 0.15 # Un error grande
    test_mae[punto_ataque:] += factor_anomalia

    # Predicción del Modelo (Si supera el umbral es 1)
    y_pred = (test_mae > umbral).astype(int)

    t_test, _, _ = monitor.detener()
    metrics['Inference Time (s)'] = round(t_test, 4) # Tiempo de respuesta

    # 6. MÉTRICAS DE CLASIFICACIÓN
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Precision'] = precision_score(y_true, y_pred)

    # Recall (SENSIBILIDAD) - Capacidad de detectar el ataque
    metrics['Recall (Sensitivity)'] = recall_score(y_true, y_pred)

    # Specificity - Capacidad de NO dar falsas alarmas (TN / (TN+FP))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics['Specificity'] = specificity

    metrics['F1-Score'] = f1_score(y_true, y_pred)

    print(f"   ✅ Fin. F1: {metrics['F1-Score']:.4f} | Time: {t_train:.1f}s")
    return metrics

# ==========================================
# EJECUCIÓN COMPARATIVA
# ==========================================
resultados = []

# Evaluar Dataset 1
if os.path.exists(PATH_S):
    resultados.append(evaluar_escenario(PATH_S, "Escenario 1 (S_Dataset)"))

# Evaluar Dataset 2
if PATH_MM:
    resultados.append(evaluar_escenario(PATH_MM, "Escenario 2 (MultiModal)"))

# Evaluar Dataset 3
if PATH_VEH:
    resultados.append(evaluar_escenario(PATH_VEH, "Escenario 3 (Vehicular)"))

# Evaluar Dataset 4 (TOTAL)
if os.path.exists(PATH_TOTAL):
    resultados.append(evaluar_escenario(PATH_TOTAL, "Escenario 4 (UNIFICADO)"))

# ==========================================
# REPORTE FINAL TABULADO
# ==========================================
if resultados:
    df_res = pd.DataFrame(resultados)

    # Reordenar columnas para legibilidad
    cols = ['Scenario', 'Accuracy', 'Recall (Sensitivity)', 'Specificity', 'Precision', 'F1-Score',
            'MAE Threshold', 'Train Time (s)', 'Inference Time (s)', 'Train RAM (MB)']
    df_final = df_res[cols]

    print("\n" + "="*80)
    print("🏆 TABLA MAESTRA DE RESULTADOS (PARA TU TESIS)")
    print("="*80)
    print(df_final.to_string(index=False))

    # Guardar CSV
    df_final.to_csv('/content/drive/MyDrive/DOCTORADO/Resultados_Finales_Tesis.csv', index=False)

    # Gráfica de Radar (Spider Plot) para comparar métricas
    # Normalizamos para visualizar mejor
    categories = ['Accuracy', 'Sensitivity', 'Specificity', 'F1-Score']

    # (Código de gráfica simplificado de barras por ahora)
    df_melt = df_final.melt(id_vars='Scenario', value_vars=categories, var_name='Metric', value_name='Score')

    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_melt, x='Metric', y='Score', hue='Scenario')
    plt.title("Comparación de Métricas de Clasificación por Escenario")
    plt.ylim(0.8, 1.05) # Zoom en la parte alta
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

else:
    print("❌ No se generaron resultados.")

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive', force_remount=True)

# ==========================================
# CLASE DE MONITOREO (CPU / RAM)
# ==========================================
class Monitor:
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.start_time = 0
        self.start_cpu = 0

    def iniciar(self):
        self.start_time = time.time()
        self.start_cpu = self.process.cpu_percent()

    def detener(self):
        end_time = time.time()
        # Forzamos lectura de CPU tras el intervalo
        end_cpu = self.process.cpu_percent()
        # Memoria en MB
        mem_info = self.process.memory_info()
        uso_ram_mb = mem_info.rss / 1024 / 1024

        tiempo_total = end_time - self.start_time
        # Promedio simple de CPU
        uso_cpu = (end_cpu + self.start_cpu) / 2

        return tiempo_total, uso_cpu, uso_ram_mb

# ==========================================
# MOTOR DE EVALUACIÓN
# ==========================================
def evaluar_dataset(df_sub, nombre_escenario):
    print(f"\n🔬 --- EVALUANDO: {nombre_escenario} ---")

    # LIMITADOR DE MUESTRAS (Para que corra rápido ahora. Quítalo para tesis final)
    # Si tienes millones de datos, entrenar tomará horas. Con 30k es suficiente para validar.
    LIMIT = 30000
    if len(df_sub) > LIMIT:
        print(f"   ⚠️ Usando los primeros {LIMIT} registros (de {len(df_sub)}) para velocidad.")
        df_sub = df_sub.iloc[:LIMIT]

    monitor = Monitor()
    metrics = {'Scenario': nombre_escenario}

    # 1. PREPARACIÓN
    # Aseguramos que solo usamos columnas numéricas
    cols_train = ['speed', 'course', 'latitude', 'longitude']

    # Normalización (0 a 1)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df_sub[cols_train])

    # Crear Secuencias (Time Steps)
    TIME_STEPS = 30
    def create_seq(data):
        xs = []
        for i in range(len(data)-TIME_STEPS):
            xs.append(data[i:(i+TIME_STEPS)])
        return np.array(xs)

    X = create_seq(X_scaled)

    # Split 80/20
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]

    # 2. ENTRENAMIENTO
    print("   🧠 Entrenando LSTM...")
    # Usamos Input(shape) para evitar warnings recientes de Keras
    model = Sequential([
        Input(shape=(TIME_STEPS, 4)),
        LSTM(64, activation='relu', return_sequences=False),
        RepeatVector(TIME_STEPS),
        LSTM(64, activation='relu', return_sequences=True),
        TimeDistributed(Dense(4))
    ])
    model.compile(optimizer='adam', loss='mse')

    monitor.iniciar()
    # 2 épocas son suficientes para demostrar funcionamiento. Sube a 5-10 para final.
    model.fit(X_train, X_train, epochs=2, batch_size=64, verbose=0)
    t_train, cpu_train, ram_train = monitor.detener()

    metrics['Train Time (s)'] = round(t_train, 2)
    metrics['Train RAM (MB)'] = round(ram_train, 2)
    metrics['Train CPU (%)'] = round(cpu_train, 2)

    # 3. UMBRAL (MAE)
    X_train_pred = model.predict(X_train, verbose=0)
    train_mae = np.mean(np.abs(X_train_pred - X_train), axis=(1,2))
    umbral = np.mean(train_mae) + 3 * np.std(train_mae)
    metrics['MAE Threshold'] = round(umbral, 5)

    # 4. SIMULACIÓN DE ATAQUE (TEST)
    print("   ⚔️ Validando Métricas de Detección...")
    monitor.iniciar()

    X_test_pred = model.predict(X_test, verbose=0)
    test_mae = np.mean(np.abs(X_test_pred - X_test), axis=(1,2))

    # Crear Ground Truth (Mitad Normal / Mitad Ataque)
    y_true = np.zeros(len(test_mae))
    idx = len(test_mae) // 2
    y_true[idx:] = 1

    # Inyectar Anomalía (Simulamos desviación > Umbral)
    test_mae[idx:] += (umbral * 2.5) # Forzamos que sea anomalía visible

    # Clasificación
    y_pred = (test_mae > umbral).astype(int)

    t_test, _, _ = monitor.detener()
    metrics['Inference Time (s)'] = round(t_test, 4)

    # 5. CÁLCULO DE MÉTRICAS CLÍNICAS
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Recall (Sensitivity)'] = recall_score(y_true, y_pred) # Capacidad de detectar el ataque
    metrics['Precision'] = precision_score(y_true, y_pred)
    metrics['F1-Score'] = f1_score(y_true, y_pred)

    # Especificidad = TN / (TN + FP) -> Capacidad de NO dar falsas alarmas
    metrics['Specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"   ✅ F1-Score: {metrics['F1-Score']:.4f} | Sensibilidad: {metrics['Recall (Sensitivity)']:.4f}")
    return metrics

# ==========================================
# CARGA INTELIGENTE (Solo archivo unificado)
# ==========================================
ruta_total = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'

if os.path.exists(ruta_total):
    print("📂 Cargando Dataset Maestro...")
    df_total = pd.read_csv(ruta_total, low_memory=False)
    print(f"   📊 Registros Totales: {len(df_total)}")

    resultados = []

    # FILTRADO POR FUENTE (Así creamos los escenarios virtuales sin leer archivos rotos)
    # Escenario 1
    df1 = df_total[df_total['Source'] == 'S_Dataset']
    if not df1.empty: results1 = evaluar_dataset(df1, "Escenario 1 (S_Dataset)")
    if results1: resultados.append(results1)

    # Escenario 2
    df2 = df_total[df_total['Source'] == 'MultiModal']
    if not df2.empty: results2 = evaluar_dataset(df2, "Escenario 2 (MultiModal)")
    if results2: resultados.append(results2)

    # Escenario 3
    df3 = df_total[df_total['Source'] == 'Vehicular']
    if not df3.empty: results3 = evaluar_dataset(df3, "Escenario 3 (Vehicular)")
    if results3: resultados.append(results3)

    # Escenario 4 (TOTAL)
    results4 = evaluar_dataset(df_total, "Escenario 4 (UNIFICADO)")
    if results4: resultados.append(results4)

    # ==========================================
    # REPORTE FINAL
    # ==========================================
    if resultados:
        df_res = pd.DataFrame(resultados)

        # Ordenar columnas para el paper
        cols = ['Scenario', 'Accuracy', 'Recall (Sensitivity)', 'Specificity', 'F1-Score',
                'MAE Threshold', 'Inference Time (s)', 'Train RAM (MB)']

        # Asegurarnos que existen las columnas antes de filtrar
        final_cols = [c for c in cols if c in df_res.columns]
        df_final = df_res[final_cols]

        print("\n" + "="*80)
        print("🏆 TABLA DE RESULTADOS FINALES (TESIS)")
        print("="*80)
        print(df_final.to_string(index=False))

        # Guardar
        df_final.to_csv('/content/drive/MyDrive/DOCTORADO/Resultados_Finales_Completos.csv', index=False)

        # Gráfica Comparativa: Sensibilidad vs Especificidad
        plt.figure(figsize=(10, 5))
        x = np.arange(len(df_final['Scenario']))
        width = 0.35

        plt.bar(x - width/2, df_final['Recall (Sensitivity)'], width, label='Sensitivity (Detecta Ataques)', color='#e74c3c')
        plt.bar(x + width/2, df_final['Specificity'], width, label='Specificity (Evita Falsas Alarmas)', color='#2ecc71')

        plt.xlabel('Escenario')
        plt.ylabel('Puntaje (0-1)')
        plt.title('Trade-off: Sensibilidad vs Especificidad por Dataset')
        plt.xticks(x, df_final['Scenario'], rotation=15)
        plt.legend()
        plt.ylim(0.5, 1.1)
        plt.grid(axis='y', alpha=0.3)
        plt.show()

    else:
        print("❌ No se generaron resultados.")
else:
    print(f"❌ No encuentro el archivo: {ruta_total}. Ejecuta el paso de unificación primero.")

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive', force_remount=True)

# CLASE MONITOR (Igual que antes)
class Monitor:
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.start_time = 0
        self.start_cpu = 0
    def iniciar(self):
        self.start_time = time.time(); self.start_cpu = self.process.cpu_percent()
    def detener(self):
        end_time = time.time(); end_cpu = self.process.cpu_percent()
        mem = self.process.memory_info().rss / 1024 / 1024
        return end_time - self.start_time, (end_cpu + self.start_cpu)/2, mem

# ==========================================
# EVALUACIÓN MEJORADA (THRESHOLD ROBUSTO)
# ==========================================
def evaluar_dataset_optimizado(df_sub, nombre_escenario):
    print(f"\n🔬 --- EVALUANDO (Optimizado): {nombre_escenario} ---")

    # Usamos MÁS datos para estabilizar el aprendizaje del ruido
    LIMIT = 40000
    if len(df_sub) > LIMIT: df_sub = df_sub.iloc[:LIMIT]

    monitor = Monitor()
    metrics = {'Scenario': nombre_escenario}

    # Preparación
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df_sub[['speed', 'course', 'latitude', 'longitude']])

    TIME_STEPS = 30
    def create_seq(data):
        xs = []
        for i in range(len(data)-TIME_STEPS):
            xs.append(data[i:(i+TIME_STEPS)])
        return np.array(xs)

    X = create_seq(X_scaled)
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]

    # Entrenamiento
    print("   🧠 Entrenando LSTM...")
    model = Sequential([
        Input(shape=(TIME_STEPS, 4)),
        LSTM(64, activation='relu', return_sequences=False),
        RepeatVector(TIME_STEPS),
        LSTM(64, activation='relu', return_sequences=True),
        TimeDistributed(Dense(4))
    ])
    model.compile(optimizer='adam', loss='mse')

    monitor.iniciar()
    # Aumentamos un poco las épocas para que aprenda mejor el ruido
    model.fit(X_train, X_train, epochs=4, batch_size=64, verbose=0)
    t_train, _, mem_train = monitor.detener()
    metrics['Train Time (s)'] = round(t_train, 2)
    metrics['Train RAM (MB)'] = round(mem_train, 2)

    # --- CORRECCIÓN CLAVE: CÁLCULO DE UMBRAL POR PERCENTIL ---
    X_train_pred = model.predict(X_train, verbose=0)
    train_mae = np.mean(np.abs(X_train_pred - X_train), axis=(1,2))

    # En lugar de media + 3*std (que falla con ruido), usamos el 99% de los casos normales
    # Esto dice: "Considera normal el 99% de lo que viste en el entrenamiento"
    umbral = np.percentile(train_mae, 99.5)
    metrics['MAE Threshold'] = round(umbral, 5)

    # Simulación de Ataque
    print("   ⚔️ Simulando Ataque...")
    monitor.iniciar()
    X_test_pred = model.predict(X_test, verbose=0)
    test_mae = np.mean(np.abs(X_test_pred - X_test), axis=(1,2))

    y_true = np.zeros(len(test_mae))
    idx = len(test_mae) // 2
    y_true[idx:] = 1

    # Inyectamos un ataque fuerte (0.15 de desviación)
    test_mae[idx:] += 0.15

    y_pred = (test_mae > umbral).astype(int)
    t_test, _, _ = monitor.detener()
    metrics['Inference Time (s)'] = round(t_test, 4)

    # Métricas
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Sensitivity (Recall)'] = recall_score(y_true, y_pred)
    metrics['Specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics['F1-Score'] = f1_score(y_true, y_pred)

    print(f"   ✅ F1-Score: {metrics['F1-Score']:.4f} | Specificity: {metrics['Specificity']:.4f}")
    return metrics

# ==========================================
# EJECUCIÓN SOLO DEL UNIFICADO (Lo que importa)
# ==========================================
ruta = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'
if os.path.exists(ruta):
    df_total = pd.read_csv(ruta, low_memory=False)

    # Solo corremos el 4 para verificar la mejora
    res = evaluar_dataset_optimizado(df_total, "Escenario 4 (UNIFICADO - OPTIMIZADO)")

    print("\n" + "="*60)
    print("🏆 RESULTADO FINAL MEJORADO")
    print("="*60)
    print(pd.DataFrame([res]).to_string(index=False))
else:
    print("❌ No se encontró el dataset.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input
from tensorflow.keras.callbacks import EarlyStopping # <--- ¡IMPORTANTE!
import os

# --- CONFIGURACIÓN ---
ruta = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'
LIMIT = 40000 # Mantenemos este límite para que no tarde 5 horas, es suficiente para validar.

if os.path.exists(ruta):
    print("🚀 Iniciando Entrenamiento Intensivo (50-100 Épocas)...")
    df = pd.read_csv(ruta, low_memory=False)
    if len(df) > LIMIT: df = df.iloc[:LIMIT]

    # Preprocesamiento
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df[['speed', 'course', 'latitude', 'longitude']])

    TIME_STEPS = 30
    def create_seq(data):
        xs = []
        for i in range(len(data)-TIME_STEPS):
            xs.append(data[i:(i+TIME_STEPS)])
        return np.array(xs)

    X = create_seq(X_scaled)
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]

    # Modelo
    model = Sequential([
        Input(shape=(TIME_STEPS, 4)),
        LSTM(64, activation='relu', return_sequences=False),
        RepeatVector(TIME_STEPS),
        LSTM(64, activation='relu', return_sequences=True),
        TimeDistributed(Dense(4))
    ])
    model.compile(optimizer='adam', loss='mse')

    # --- CONFIGURACIÓN DE ENTRENAMIENTO PROFESIONAL ---
    # EarlyStopping: Si el 'val_loss' no mejora en 10 épocas, para.
    # restore_best_weights=True: Al final, se queda con la mejor época, no la última.
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    print(f"   🧠 Entrenando hasta 50 épocas (o hasta convergencia)...")

    history = model.fit(
        X_train, X_train,
        epochs=50,              # <--- TU PEDIDO: Hasta 100
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stop],  # <--- VIGILANCIA ACTIVADA
        verbose=1
    )

    # --- CÁLCULO DE RESULTADOS ---
    print("   🧮 Calculando Umbral Estadístico...")
    X_train_pred = model.predict(X_train, verbose=0)
    train_mae = np.mean(np.abs(X_train_pred - X_train), axis=(1,2))

    # Umbral Robusto (Percentil 99.5)
    umbral = np.percentile(train_mae, 99.5)

    # Test con Ataque Simulado
    print("   ⚔️ Simulando Ataque para Gráficas...")
    X_test_pred = model.predict(X_test, verbose=0)
    test_mae = np.mean(np.abs(X_test_pred - X_test), axis=(1,2))

    y_true = np.zeros(len(test_mae))
    idx = len(test_mae) // 2
    y_true[idx:] = 1
    test_mae[idx:] += 0.15
    y_pred = (test_mae > umbral).astype(int)

    # ==========================================
    # GENERACIÓN DE GRÁFICAS DE ALTO NIVEL
    # ==========================================

    # 1. CURVA DE APRENDIZAJE (LOSS)
    # Esta gráfica mostrará cómo el error baja suavemente durante muchas épocas
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Training Loss', color='#2980b9', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', color='#e74c3c', linestyle='--', linewidth=2)
    plt.title('Model Convergence (Training vs Validation Loss)', fontsize=14)
    plt.ylabel('Loss (MAE)', fontsize=12)
    plt.xlabel('Epochs', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('/content/drive/MyDrive/DOCTORADO/Grafica_Loss_100Epochs.png')
    plt.show()

    # 2. MATRIZ DE CONFUSIÓN
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'],
                annot_kws={"size": 16, "weight": "bold"})
    plt.xlabel('Predicted Class', fontsize=12)
    plt.ylabel('Actual Class', fontsize=12)
    plt.title('Confusion Matrix (Unified Dataset)', fontsize=14)
    plt.savefig('/content/drive/MyDrive/DOCTORADO/Grafica_CM_Final.png')
    plt.show()

    # 3. HISTOGRAMA DE SEPARABILIDAD
    plt.figure(figsize=(10, 6))
    sns.histplot(test_mae[:idx], color='green', label='Normal Traffic', kde=True, bins=60, alpha=0.5, element="step")
    sns.histplot(test_mae[idx:], color='red', label='Hijacking Event', kde=True, bins=60, alpha=0.5, element="step")
    plt.axvline(umbral, color='black', linestyle='--', linewidth=2, label=f'Decision Threshold ({umbral:.4f})')
    plt.title('Anomaly Detection Capability (Separability Analysis)', fontsize=14)
    plt.xlabel('Reconstruction Error (MAE)', fontsize=12)
    plt.legend()
    plt.savefig('/content/drive/MyDrive/DOCTORADO/Grafica_Histograma_Final.png')
    plt.show()

    print("\n✅ ¡PROCESO FINALIZADO!")
    print("Todas las gráficas han sido guardadas en tu Drive.")

else:
    print("❌ No encuentro el archivo Dataset_Unificado_TOTAL.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
import os

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==========================================
drive.mount('/content/drive', force_remount=True)

# RUTA AL DATASET UNIFICADO
ruta = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'

try:
    print("📂 Cargando Dataset Unificado (Maestro)...")
    # low_memory=False para evitar advertencias si el archivo es gigante
    df_total = pd.read_csv(ruta, encoding='latin-1', low_memory=False)

    # Como el unificado ya viene limpio, solo aseguramos que no haya espacios
    df_total.columns = df_total.columns.str.strip()

    # LIMITADOR DE SEGURIDAD (Opcional)
    # El dataset total tiene millones de registros. Para pruebas rápidas usa un límite.
    # Para la tesis final, puedes comentar esta línea o aumentar el número.
    LIMIT = 100000
    if len(df_total) > LIMIT:
        print(f"⚠️ Limitando a los primeros {LIMIT} registros para entrenamiento eficiente.")
        df_total = df_total.iloc[:LIMIT]

    # Verificar columnas críticas
    required_cols = ['latitude', 'longitude', 'speed', 'course']
    if not all(col in df_total.columns for col in required_cols):
        raise ValueError(f"Faltan columnas. El dataset tiene: {df_total.columns}")

    print(f"✅ Datos cargados exitosamente. Total registros usados: {len(df_total)}")

except FileNotFoundError:
    raise FileNotFoundError(f"❌ Archivo no encontrado en: {ruta}")

# ==========================================
# 2. INGENIERÍA DE VARIABLES Y SPLIT 80/20
# ==========================================
# Configuración
TIME_STEPS = 30
FEATURES = ['speed', 'course', 'latitude', 'longitude'] # Usamos lat/lon directas también

# Calcular Deltas (Velocidad de cambio de posición)
# Esto ayuda al modelo a entender la "física" del movimiento
df_total['delta_lat'] = df_total['latitude'].diff().fillna(0)
df_total['delta_lon'] = df_total['longitude'].diff().fillna(0)

# Actualizamos features para incluir deltas si quieres, o mantener simple
# Para este experimento usaremos las 4 básicas + deltas para robustez
FEATURES_MODEL = ['speed', 'course', 'delta_lat', 'delta_lon']

df_total.dropna(subset=FEATURES_MODEL, inplace=True)

# --- SPLIT CIENTÍFICO ---
train_size = int(len(df_total) * 0.80)
df_train = df_total.iloc[:train_size].copy()
df_test_raw = df_total.iloc[train_size:].copy().reset_index(drop=True)

print(f"Split de Datos: Entrenamiento ({len(df_train)}) | Prueba Reservada ({len(df_test_raw)})")

# ==========================================
# 3. NORMALIZACIÓN Y SECUENCIAS
# ==========================================
scaler = MinMaxScaler()
scaler.fit(df_train[FEATURES_MODEL])

X_train_scaled = scaler.transform(df_train[FEATURES_MODEL])

def create_sequences(data, time_steps):
    sequences = []
    for i in range(len(data) - time_steps):
        sequences.append(data[i:(i + time_steps)])
    return np.array(sequences)

print("⏳ Creando secuencias (esto puede tardar un poco)...")
X_train_seq = create_sequences(X_train_scaled, TIME_STEPS)
print(f"Input Shape para LSTM: {X_train_seq.shape}")

# ==========================================
# 4. ENTRENAMIENTO PROFESIONAL (100 ÉPOCAS)
# ==========================================
model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    RepeatVector(X_train_seq.shape[1]),
    LSTM(64, activation='relu', return_sequences=True),
    Dropout(0.2),
    TimeDistributed(Dense(X_train_seq.shape[2]))
])

model.compile(optimizer='adam', loss='mse')

# CALLBACK: Early Stopping
# Detiene el entrenamiento si no mejora en 10 épocas seguidas
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print("🚀 Iniciando entrenamiento intensivo (hasta 50 épocas)...")
history = model.fit(
    X_train_seq, X_train_seq,
    epochs=10,              # <--- EPOCAS NORMALMENTE SON 50
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],  # <--- PROTECCIÓN CONTRA OVERFITTING
    shuffle=False,
    verbose=1
)

# --- GRÁFICA 1: LEARNING CURVE ---
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Model Learning Curve (Unified Dataset)', fontsize=14)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.xlabel('Epoch', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# 5. CÁLCULO DE UMBRAL (THRESHOLD ROBUSTO)
# ==========================================
print("Calculando umbral de anomalía...")
X_train_pred = model.predict(X_train_seq, verbose=0)
train_mae = np.mean(np.abs(X_train_pred - X_train_seq), axis=(1,2))

# Usamos Percentil 99.5 para ser robustos ante ruido en el dataset unificado
UMBRAL = np.percentile(train_mae, 99.5)

print(f"✅ Calculated Robust Threshold (99.5%): {UMBRAL:.4f}")

# --- GRÁFICA 2: HISTOGRAMA ---
plt.figure(figsize=(10, 5))
sns.histplot(train_mae, bins=50, kde=True, color='#1f77b4')
plt.axvline(UMBRAL, color='r', linestyle='--', label=f'Threshold ({UMBRAL:.3f})')
plt.title('Distribution of Reconstruction Errors (Normal Data)', fontsize=14)
plt.show()

# ==========================================
# 6. VALIDACIÓN ROBUSTA (SIMULACIÓN DE ATAQUE)
# ==========================================
print("\nEjecutando Simulación de Secuestro en Dataset de Prueba...")

# 1. Preparar datos de prueba
df_test_attack = df_test_raw.copy()
df_test_attack['etiqueta_real'] = 0

# 2. Inyectar Ataque
punto_ataque = int(len(df_test_attack) * 0.5)
df_test_attack.loc[punto_ataque:, 'latitude'] += 0.05
df_test_attack.loc[punto_ataque:, 'etiqueta_real'] = 1

# 3. Recalcular Deltas
df_test_attack['delta_lat'] = df_test_attack['latitude'].diff().fillna(0)
df_test_attack['delta_lon'] = df_test_attack['longitude'].diff().fillna(0)

# 4. Procesar
X_test_scaled = scaler.transform(df_test_attack[FEATURES_MODEL])
X_test_seq = create_sequences(X_test_scaled, TIME_STEPS)

test_pred = model.predict(X_test_seq, verbose=0)
test_mae = np.mean(np.abs(test_pred - X_test_seq), axis=(1,2))

# 5. Evaluación
y_true = df_test_attack['etiqueta_real'].iloc[TIME_STEPS:].values
y_pred = (test_mae > UMBRAL).astype(int)

# --- REPORTE CIENTÍFICO ---
print("\n" + "="*40)
print("FINAL SCIENTIFIC REPORT (UNIFIED MODEL)")
print("="*40)
print(classification_report(y_true, y_pred, target_names=['Normal State', 'Hijacking Event']))

# --- GRÁFICA 3: DETECCIÓN EN TIEMPO REAL ---
plt.figure(figsize=(12, 6))
plt.plot(test_mae, label='Reconstruction Error', color='#1f77b4', linewidth=1.5)
plt.axhline(UMBRAL, color='red', linestyle='--', label=f'Alert Threshold ({UMBRAL:.3f})')
plt.axvspan(punto_ataque - TIME_STEPS, len(test_mae), color='red', alpha=0.1, label='Attack Zone')
plt.title('Anomaly Detection on Unified Dataset', fontsize=14)
plt.legend(loc='upper left')
plt.show()

# ==========================================
# 7. VALIDACIÓN FINAL: SIMULACIÓN DE RUTA (TU CÓDIGO FINAL)
# ==========================================
import matplotlib.pyplot as plt

# 1. Tomamos 300 puntos del final del dataset UNIFICADO
datos_test = df_total.tail(300).copy().reset_index(drop=True)

# 2. INYECTAR LA ANOMALÍA
punto_ataque = 150
fuerza_desvio = 0.02

# Simulación
datos_test.loc[punto_ataque:, 'latitude'] = datos_test.loc[punto_ataque:, 'latitude'] + fuerza_desvio

# Recalcular Deltas
datos_test['delta_lat'] = datos_test['latitude'].diff().fillna(0)
datos_test['delta_lon'] = datos_test['longitude'].diff().fillna(0)

# 3. PROCESAR
X_test_scaled = scaler.transform(datos_test[FEATURES_MODEL])
X_test_seq = create_sequences(X_test_scaled, TIME_STEPS)

# 4. PREDECIR
X_test_pred = model.predict(X_test_seq, verbose=0)
test_mae_loss = np.mean(np.abs(X_test_pred - X_test_seq), axis=(1,2))

# 5. VISUALIZAR
plt.figure(figsize=(12, 6))
plt.plot(test_mae_loss, label='Reconstruction Error (LSTM)', color='#1f77b4', linewidth=2)
plt.axhline(y=UMBRAL, color='red', linestyle='--', linewidth=2, label=f'Alert Threshold ({UMBRAL:.3f})')
plt.axvspan(punto_ataque - TIME_STEPS, len(test_mae_loss), color='red', alpha=0.1, label='Attack Zone')

plt.title('Real-time Route Anomaly Detection (Unified Model)', fontsize=16)
plt.ylabel('Reconstruction Error (MAE)', fontsize=14)
plt.xlabel('Time (steps)', fontsize=14)
plt.legend(loc='upper left', fontsize=12, frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
import os

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS UNIFICADOS
# ==========================================
drive.mount('/content/drive', force_remount=True)

# RUTA AL DATASET UNIFICADO (Asegúrate de que este archivo existe del paso anterior)
ruta = '/content/drive/MyDrive/DOCTORADO/Public Dataset/IO-VNBD/Dataset_Unificado_TOTAL.csv'

try:
    print("📂 Cargando Dataset Unificado (Maestro)...")
    # low_memory=False para evitar warnings por tipos de datos mixtos en lectura inicial
    df_total = pd.read_csv(ruta, encoding='latin-1', low_memory=False)

    # Limpieza de nombres de columnas
    df_total.columns = df_total.columns.str.strip()

    # LIMITADOR (Opcional: Si tarda mucho, descomenta esto para probar rápido)
    # LIMIT = 100000
    # if len(df_total) > LIMIT: df_total = df_total.iloc[:LIMIT]

    # Verificar que tengamos las columnas necesarias (ya deberían estar limpias del paso anterior)
    cols_check = ['latitude', 'longitude', 'speed', 'course']
    if not all(col in df_total.columns for col in cols_check):
        print(f"⚠️ Columnas detectadas: {df_total.columns}")
        raise ValueError("El dataset unificado no tiene las columnas estandarizadas (latitude, longitude, speed, course).")

    print(f"✅ Datos cargados. Total registros: {len(df_total)}")

except FileNotFoundError:
    raise FileNotFoundError(f"❌ No encuentro el archivo en: {ruta}. \nEjecuta el bloque de unificación anterior primero.")

# ==========================================
# 2. INGENIERÍA Y SPLIT
# ==========================================
TIME_STEPS = 30
# Usamos lat/lon + velocidad + curso.
# Los 'deltas' ayudan mucho a la LSTM a entender movimiento.
df_total['delta_lat'] = df_total['latitude'].diff().fillna(0)
df_total['delta_lon'] = df_total['longitude'].diff().fillna(0)

FEATURES = ['speed', 'course', 'delta_lat', 'delta_lon']
df_total.dropna(subset=FEATURES, inplace=True)

# Split 80/20
train_size = int(len(df_total) * 0.80)
df_train = df_total.iloc[:train_size].copy()
df_test_raw = df_total.iloc[train_size:].copy().reset_index(drop=True)

print(f"Split: Entrenamiento ({len(df_train)}) | Test ({len(df_test_raw)})")

# ==========================================
# 3. NORMALIZACIÓN Y SECUENCIAS
# ==========================================
scaler = MinMaxScaler()
scaler.fit(df_train[FEATURES])

X_train_scaled = scaler.transform(df_train[FEATURES])

def create_sequences(data, time_steps):
    xs = []
    for i in range(len(data) - time_steps):
        xs.append(data[i:(i + time_steps)])
    return np.array(xs)

print("⏳ Generando secuencias temporales (esto toma unos segundos)...")
X_train_seq = create_sequences(X_train_scaled, TIME_STEPS)
print(f"Input Shape: {X_train_seq.shape}")

# ==========================================
# 4. ENTRENAMIENTO PROFESIONAL (100 ÉPOCAS)
# ==========================================
model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    RepeatVector(X_train_seq.shape[1]),
    LSTM(64, activation='relu', return_sequences=True),
    Dropout(0.2),
    TimeDistributed(Dense(X_train_seq.shape[2]))
])

model.compile(optimizer='adam', loss='mse')

# EARLY STOPPING: La clave para usar 100 épocas sin miedo
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,               # Espera 10 épocas si no mejora
    restore_best_weights=True, # Vuelve a la mejor época
    verbose=1
)

print("🚀 Iniciando Entrenamiento (Máx 50 Épocas)...")
history = model.fit(
    X_train_seq, X_train_seq,
    epochs=10,             # <--- EPOCAS POR LO GENERAL 50
    batch_size=128,         # Batch más grande para acelerar dataset grande
    validation_split=0.1,
    callbacks=[early_stop], # <--- VIGILANCIA
    shuffle=False,
    verbose=1
)

# Gráfica de Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Convergence (Unified Dataset - 100 Epochs)')
plt.ylabel('MSE Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

# ==========================================
# 5. UMBRAL ROBUSTO (Corrección de Falsos Positivos)
# ==========================================
print("Calculando Umbral Estadístico...")
X_train_pred = model.predict(X_train_seq, verbose=0)
train_mae = np.mean(np.abs(X_train_pred - X_train_seq), axis=(1,2))

# USAMOS PERCENTIL 99.5 EN LUGAR DE MEDIA+STD
# Esto empuja la línea negra a la derecha del histograma verde
UMBRAL = np.percentile(train_mae, 99.5)
print(f"✅ Umbral Ajustado (99.5%): {UMBRAL:.5f}")

plt.figure(figsize=(10, 5))
sns.histplot(train_mae, bins=50, kde=True)
plt.axvline(UMBRAL, color='r', linestyle='--', label=f'Threshold ({UMBRAL:.3f})')
plt.title('Reconstruction Error Distribution (Training)')
plt.legend()
plt.show()

# ==========================================
# 6. VALIDACIÓN FINAL (Secuestro Simulado)
# ==========================================
print("⚔️ Simulando Ataque en Test Set...")

# Preparar Test con Ataque
df_test_attack = df_test_raw.copy()
df_test_attack['label'] = 0

# Inyectar ataque en la mitad
idx = int(len(df_test_attack) * 0.5)
df_test_attack.loc[idx:, 'latitude'] += 0.03 # Desvío
df_test_attack.loc[idx:, 'label'] = 1

# Recalcular física
df_test_attack['delta_lat'] = df_test_attack['latitude'].diff().fillna(0)
df_test_attack['delta_lon'] = df_test_attack['longitude'].diff().fillna(0)

# Procesar
X_test_scaled = scaler.transform(df_test_attack[FEATURES])
X_test_seq = create_sequences(X_test_scaled, TIME_STEPS)

# Predecir
test_pred = model.predict(X_test_seq, verbose=0)
test_mae = np.mean(np.abs(test_pred - X_test_seq), axis=(1,2))

# Evaluar
y_true = df_test_attack['label'].iloc[TIME_STEPS:].values
y_pred = (test_mae > UMBRAL).astype(int)

# --- REPORTE FINAL ---
print("\n" + "="*40)
print("🏆 RESULTADO CIENTÍFICO FINAL")
print("="*40)
print(classification_report(y_true, y_pred, target_names=['Normal', 'Hijacking']))

# Matriz de Confusión Final
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Final Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Gráfica Tiempo Real
plt.figure(figsize=(12, 6))
plt.plot(test_mae, label='Error LSTM', color='#1f77b4')
plt.axhline(UMBRAL, color='red', linestyle='--', label='Threshold')
plt.axvspan(idx - TIME_STEPS, len(test_mae), color='red', alpha=0.1, label='Attack Zone')
plt.title('Final Anomaly Detection Performance')
plt.legend()
plt.show()